#COMPSCI 546: Applied Information Retrieval - Spring 2026 ([website](https://groups.cs.umass.edu/zamani/compsci-546-applied-information-retrieval-spring-2026/))
##Assignment 1: Information Retrieval Metrics (Total : 100 points)

**Description**

This is a coding assignment where you will write and execute code to evaluate ranked outputs generated by a retrieval model or a recommender system. Basic proficiency in Python is recommended.  

**Instructions**

* To start working on the assignment, you would first need to save the notebook to your local Google Drive. For this purpose, you can click on *Copy to Drive* button. You can alternatively click the *Share* button located at the top right corner and click on *Copy Link* under *Get Link* to get a link and copy this notebook to your Google Drive.  

*   For questions with descriptive answers, please replace the text in the cell which states "Enter your answer here!" with your answer. If you are using mathematical notation in your answers, please define the variables.
*   You should implement all the functions yourself and should not use a library or tool for computing the metrics.
*   For coding questions, you can add code where it says "enter code here" and execute the cell to print the output.
* To create the final pdf submission file, execute *Runtime->RunAll* from the menu to re-execute all the cells and then generate a PDF using *File->Print->Save as PDF*. Make sure that the generated PDF contains all the codes and printed outputs before submission. You are responsible for uploading the correct pdf with all the information required for grading.
To create the final python submission file, click on File->Download .py.


**Submission Details**

* Due data: Feb. 23, 2026 at 11:59 PM (EST).
* The final PDF must be uploaded on Gradescope.
* After copying this notebook to your Google Drive, please paste a link to it below. Use the same process given above to generate a link. ***You will not recieve any credit if you don't paste the link!*** Make sure we can access the file.
***LINK: https://colab.research.google.com/drive/1i6QEvN_M5J22G2q52djS169ETHorA_bo?usp=sharing***

**Academic Honesty**

Please follow the guidelines under the *Collaboration and Help* section of the course website.     

# Download input files

Please execute the cell below to download the input files.

In [ ]:

!pip install PyDrive2

import os
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)


import os
import zipfile

download = drive.CreateFile({'id': '1myaSouVnJygjLlQI54_S0vRXLNYekEC8'})
download.GetContentFile('HW01.zip')

with zipfile.ZipFile('HW01.zip', 'r') as zip_file:
    zip_file.extractall('./')
os.remove('HW01.zip')
# We will use hw1 as our working directory
os.chdir('HW01')

#Setting the input files
qrel_file = "antique-train-final.qrel"
rank_file = "ranking_file"

# 1: Initial Data Setup (10 Points)

We use the files from the ANTIQUE dataset [https://arxiv.org/pdf/1905.08957.pdf] for this assignment. This is a passage retrieval dataset for non-factoid questions created by the Center for Intelligent Information Retrieval (CIIR) at UMass Amherst.

The description of the input files provided for this assignment is given below.

**1) Query Relevance (qrel) file**

The qrel file contains the relevance judgements (ground truth) for the query passage combinations. The file consists of 4 columns with the information given below.

*\[queryid]  [topicid]  [passageid]  [relevancejudgment]*

Entries in each row are space separated. The second column (topicid) can be ignored.

Given below are a couple of rows of a sample qrel file.

*2146313 U0 2146313_0 4*

*2146313 Q0 2146313_23 2*

The relevance judgements range from values 1-4.
The description of the labels is given below:

Label 1: Non-Relevant

Label 2: Slightly Relevant

Label 3 : Relevant

Label 4: Highly Relevant

**Note**: that for metrics with binary relevance assumptions, Labels 1 and 2 are considered non-relevant and Labels 3 and 4 are considered relevant.

**Note**: if a query-document pair is not listed in the qrels file, we assume that the document is not relevant to the query.

**2) Ranking file**

The evaluation metric value has to be calculated for the input ranking file. The file was generated using a standard search engine by executing a ranking model, retrieving the top 100 passages for each of the train queries of the ANTIQUE dataset. The format of this file is given below.

*\[queryid]  [topicid]  [passageid]  [rank] [relevance_score]  [indri]*

Similar to the qrel file, the entries in each row are space delimited.

Given below are some sample examples of the ranking file contents.

*3097310 Q0 2367043_3 1 -6.01785 indri*

*3097310 Q0 3007432_0 2 -6.22531 indri*

*3097310 Q0 674672_2 3 -6.28514 indri*

**Note**: For this assignment, you would only need the information from Column 1(queryid) and Column 3(passageid). The passages corresponding to each query is ranked with respect to the relevance score (highest to lowest), therefore you would not need to use Column 4 (rank) explicitly.




In order to make it easier to access this information in subsequent cells, please store them in appropriate data structures in the cell below.

In [ ]:

from collections import defaultdict
'''
In this function, load the qrel file into qrel datastructure
Return Variables:
num_queries_1 - Number of unique queries in the qrel file
num_rel - Number of total relevant passages in the dataset across all queries
qrels - datastructure with the query passage relevance information
'''
def loadQrels(qrel_file):
    # Define data structure
    num_queries_1 = 0
    num_rel = 0
    qrels = defaultdict(dict)

    with open(qrel_file) as f:
      for line in f:
        # Extracting info on query - passage id - relevant score
        queryInfo = line.strip().split()
        query_id = queryInfo[0]
        passage_id = queryInfo[2]
        rel_score = int(queryInfo[3])

        # If the query - passage is relevant
        if passage_id not in qrels[query_id]:
          if rel_score >= 2:
            num_rel += 1

        # We store query id (key): list of (passage id, relevant score) (value)
        qrels[query_id][passage_id] = rel_score

    # update num_queries_1 variable
    num_queries_1 = len(qrels)

    return num_queries_1, num_rel, qrels


'''
In this function, load the ranking files into rank_in datastructure
Return Variables:
num_queries_2 - Number of unique queries in the ranking file
rank_in - datastructure with stored ranking information
'''
def  loadRankfile(rank_file):
    # Define data structure
    num_queries_2 = 0
    rank_in = defaultdict(list)

    with open(rank_file) as f:
      for line in f:
        # [queryid] [topicid] [passageid] [rank] [relevance_score] [indri]
        # Extrating info on the above field
        queryInfo = line.strip().split()
        query_id = queryInfo[0]
        passage_id = queryInfo[2]
        rank_in[query_id].append(passage_id)

    num_queries_2 = len(rank_in)

    return num_queries_2, rank_in



''' You can return single/multiple variables to store data if that makes it convenient
for data processing.
This has been given as an example. However, you would still need to correctly print the
queries in both files and total relevant passages.'''
num_queries_1, num_rel, qrels  = loadQrels(qrel_file)
num_queries_2, rank_in = loadRankfile(rank_file)

# print to ensure the file has been read correctly
print ('Total Num of queries in the qrel file  : {0}'.format(num_queries_1))
print ('Total Num of queries in the rank file  : {0}'.format(num_queries_2))
print ('Total number of relevant passages in the dataset :{0}'.format(num_rel))

Total Num of queries in the qrel file  : 2426
Total Num of queries in the rank file  : 2426
Total number of relevant passages in the dataset :26150




# 2 : Precision (15 Points)


Question 2.1 (5 points)

Give the definition of Precision corresponding to the top *k* retrieved results for *n* queries (P@k). Please note that you have to use averaging to aggregate the results from all queries.   

**Answer:**\
Precision@k is the average, over all the queries, of the proportion of relevant documents among the top k retreived documents for each query.
$$P@k = \frac{1}{n}\sum^{n}_{i = 1}P_{q_i}(k)$$
$$P_q(k) = \frac{|\text{relevant}_q \cap \text{Top-}k_q|}{k}$$
where
  * n is the total number of queries
  * $P_{q_i}(k)$ is the precision at rank $k$ for query $q_i$



Question 2.2 (10 points)

In the cell below, please enter the code to print the P@k where k={5,10} for the input ranking file.  As mentioned above, please note that the final value is the average of metric values across all queries.


In [ ]:
'''
In this function, calculate Precision@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
precision - Precision@k
'''
def calcPrecision(k, qrels, rank_in):
  # Data Structure Initialization
  precisionArr = [0] * len(rank_in)

  for idx, query in enumerate(rank_in):
    passages = rank_in[query]
    passagesRank = qrels[query]
    numRelevant = 0
    for i in range(k): # highly assume we always have more than k passages
      if passages[i] in passagesRank and passagesRank[passages[i]] >= 3:
        numRelevant += 1

    precisionArr[idx] = numRelevant / k

  precision = sum(precisionArr) / len(precisionArr)

  return precision

print ('Precision at top 5 : {0}'.format(calcPrecision(5, qrels, rank_in)))
print ('Precision at top 10 : {0}'.format(calcPrecision(10, qrels, rank_in)))



Precision at top 5 : 0.15539983511953834
Precision at top 10 : 0.1072959604286892


# 3 : Recall (15 points)

Question 3.1 (5 points)

Give the definition of Recall corresponding to the top *k* retrieved results for *n* queries (R@k). Please note that you have to use averaging to aggregate the results from all queries.

**Answer:**\
For a query $q$, Recall at rank $k$ is defined as:\
$$R@k(q) = \frac{\{\text{relevant documents retrieved in top k}\}}{\{\text{all relevant documents for q}\}}$$\
The overall Recall@k is computed by averaging over all queries:\
$$R@k = \frac{1}{n}\sum^{n}_{i = 1}R@k(q_i)$$

Question 3.2 (10 points)

In the cell below, please enter the code to print Recall (R@k) where k={5,10} for the input ranking file. As mentioned above, please note that the final value is the average of metric values across all queries.

In [ ]:
'''
In this function, calculate Recall@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
recall - Recall@k
'''
def calcRecall(k, qrels, rank_in):
    #enter your code here
    recallArray = [0] * len(rank_in)

    for idx, query in enumerate(rank_in):
      passages = rank_in[query] # array containing all of the passage retrieved (that the machine think it is relevant)
      passagesRank = qrels[query] # dictionary containing all of the passage -> relevance score
      numRelevant = 0
      totalRelevant = 0

      # It has to be number of relevant document according to
      # all of the relevant document using that exact query!

      # We first retrieve k element from retrieved list:
      k_passages = set(passages[:k])
      for passage, relevanceScore in passagesRank.items():
        # check if it is relevant or not
        if relevanceScore >= 3:
          totalRelevant += 1
        if passage in k_passages and relevanceScore >= 3:
          numRelevant += 1

      recallArray[idx] = numRelevant / totalRelevant

    recall = sum(recallArray) / len(recallArray)

    return recall

print ('Recall at top 5 : {0}'.format(calcRecall(5, qrels, rank_in)))
print ('Recall at top 10 : {0}'.format(calcRecall(10, qrels, rank_in)))

Recall at top 5 : 0.12934416493801082
Recall at top 10 : 0.16743513404735452


# 4 : F1 Measure (15 Points)

Question 4.1 (5 points)

Give the definition of F1 measure corresponding to the top *k* retrieved results for *n* queries (F1@k). Please note that you have to use averaging to aggregate the results from all queries.

**Answer:**\
F1 measure at rank $k$ for query $q$ is defined as the harmonic mean of $P@k(q)$ and $R@k(q)$\
$$F1@k(q) = \frac{2\cdot P@k(q)\cdot R@k(q)}{P@k(q) + R@k(q)}$$

Question 4.2 (10 points)

In the cell below, please enter the code to print the F1@k where k={5,10} for the input ranking file.  Please note that you have to calculate F1 score for each query and compute the final score by averaging the metric values across all queries.

In [ ]:
'''
In this function, calculate F1@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
f1 - F1@k
'''

def calcFScore(k, qrels, rank_in):
    # Precalc
    precision_arr = [0] * len(rank_in)
    recall_arr = [0] * len(rank_in)

    for idx, query in enumerate(rank_in):
      # Calculation of precision for a single query
      passages = rank_in[query]
      passage_rank = qrels[query]
      num_relevant = 0

      for passage in passages[:k]:
        if passage in passage_rank and passage_rank[passage] >= 3:
          num_relevant += 1

      precision_arr[idx] = num_relevant / k

      # Calculation of recall for a single query:
      total_relevant = 0
      for passage, relevance_score in passage_rank.items():
        if relevance_score >= 3:
          total_relevant += 1

      recall_arr[idx] = num_relevant / total_relevant

    # Calculation of f1 for each query
    f1_array = [0] * len(rank_in)

    for i in range(len(precision_arr)):
      precision_value_i = precision_arr[i]
      recall_value_i = recall_arr[i]
      if precision_value_i + recall_value_i != 0:
        f1_array[i] = 2 * precision_value_i * recall_value_i / (precision_value_i + recall_value_i)

    f1 = sum(f1_array) / len(f1_array)

    return f1

print ('F1 score at top 5 : {0}'.format(calcFScore(5, qrels, rank_in)))
print ('F1 score at top 10 : {0}'.format(calcFScore(10, qrels, rank_in)))

F1 score at top 5 : 0.125871711807863
F1 score at top 10 : 0.11669190136577237


# 5 : Mean Reciprocal Rank (MRR) (15 Points)

Question 5.1 (5 points)

Give the definition of MRR@k corresponding to the top *k* retrieved results for *n* queries (MRR@k). Please note that you have to use averaging to aggregate the results from all queries.

**Answer:**\
For a query $q$, let $r_q$ be the rank position (within the top $k$ retrieved results) of the first relevant document (where relevance label $\geq 3$).\
The Reciprocal Rank at cutoff $k$ is defined as:\
$$RR@k(q) = \begin{cases}
  \frac{1}{r_q}\,, \text{if a relevant document appears within the top $k$}\\
  0\,, \text{otherwise}
\end{cases}$$\
**Aggregation over $n$ queries**\
The Mean Reciprocal Rank at cutoff $k$ is computed by averaging over all queries:\
$$MRR@k = \frac{1}{n}\sum_{i=1}^{n}RR@k(q_i)$$

Question 5.2 (10 points)

In the cell below, please enter the code to print the MRR@k where k={5,10} for the input ranking file. As mentioned above, please note that the final value is the average of metric values across all queries.

In [ ]:
'''
In this function, calculate MRR@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
mrr - MRR@k
'''

def calcMRR(k, qrels, rank_in):
     #enter your code here
     reciprocal_array = [0] * len(rank_in)
     for idx, query in enumerate(rank_in):
       passages = rank_in[query]
       passage_rank = qrels[query]

       for idxPassage, passage in enumerate(passages[:k]):
         if passage in passage_rank and passage_rank[passage] >= 3:
          reciprocal_array[idx] = 1 / (idxPassage + 1)
          break

     mrr = sum(reciprocal_array) / len(reciprocal_array)

     return mrr

print ('MRR at top 5 : {0}'.format(calcMRR(5, qrels, rank_in)))
print ('MRR at top 10 : {0}'.format(calcMRR(10, qrels, rank_in)))

MRR at top 5 : 0.3375996152789228
MRR at top 10 : 0.3474898258286552


# 6 : Mean Average Precision (MAP) (15 points)

Question 6.1 (5 points)

Give the definition of MAP@k corresponding to the top *k* retrieved results for *n* queries (MAP@k). Please note that you have to use averaging to aggregate the results from all queries.

**Answer:**\
Mean Average Precision at cutoff $k$ measures the average of the Average Precision values computed for each query, considering only the top $k$ retrieved document.\
For each query, Average Precision evaluates how well relevant documents are ranked within the top $k$ positions by averaging the precision values computed at the ranks where relevant documents occur.\
$$AP@k(q) = \frac{1}{R_q}\sum_{i=1}^{k}P@i(q)\cdot \text{rel}_1(i)$$
where
$$P@i(q) = \frac{\text{# relevant documents in top }i}{i}$$\
**Mean Average Precision at cutoff $k$**\
$$MAP@k = \frac{1}{n}\sum_{j=1}^{n}AP@k(q_j)$$

Question 6.2 (10 points)

In the cell below, please enter the code to print the MAP@k where k={50, 100} for the input ranking file. As mentioned above, please note that the final value is the average of metric values across all queries.


In [ ]:
'''
In this function, calculate MAP@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
map - MAP@k
'''

def calcMAP(k, qrels, rank_in):
  # Data structure init
  ap_array = [0] * len(rank_in)

  for idx, query in enumerate(rank_in):
    # know how many relevant passage with that query
    passages = rank_in[query]
    passage_rank = qrels[query]
    num_relevant = 0 # denote number of relevancy with that query
    for passage in passage_rank:
      if passage in passage_rank and passage_rank[passage] >= 3:
        num_relevant += 1

    # calculate precision:
    cnt_relevant = 0
    sumPrec = 0
    for idxPassage, passage in enumerate(passages[:k]):
      if passage in passage_rank and passage_rank[passage] >= 3:
        cnt_relevant += 1
        sumPrec += cnt_relevant / (idxPassage + 1)

    ap_array[idx] = sumPrec / num_relevant

  map = sum(ap_array) / len(ap_array)

  return map

print ('MAP at top 50 : {0}'.format(calcMAP(50, qrels, rank_in)))
print ('MAP at top 100 : {0}'.format(calcMAP(100, qrels, rank_in)))

MAP at top 50 : 0.12134118627027753
MAP at top 100 : 0.12327733804549047


# 7 : Normalized Discounted Cumulative Gain (NDCG) (15 Points)

Question 7.1 (5 points)

Give the definition of NDCG@k corresponding to the top *k* retrieved results for *n* queries (NDCG@k). Use the definition discussed in the lectures. Note that this metric considers graded relevance judgments and you should not binarize the labels. To assign zero gain to non-relevant documents, decrease all relevance labels in the ANTIQUE qrels by 1 point i.e. map relevance judgements 1-4 to 0-3. Please note that you have to use averaging to aggregate the results from all queries.

**Answer:**\
NDCG at cutoff $k$ evaluates ranking quality using graded relevance scores. It measures how close the ranking is to the ideal ranking (sorted by highest relevance list), considering position-based discounting.\
\
**DCG@k**
$$DCG@k = \sum_{i=1}^{k}\frac{g_i}{\log_2(i + 1)}$$\
\
**IDCG@k**
$$IDCG@k = \sum_{i = 1}^{k}\frac{g_i^*}{\log_2(i + 1)}$$\
\
**NDCG@k**
$$NDCG@k = \frac{DCG@k}{IDCG@k}$$\
If $IDCG@k = 0$, then:
$$NDCG@k = 0$$\
**Aggregation over $n$ queries**
$$NDCG@k = \frac{1}{n}\sum_{j = 1}^{n}NDCG@k(q_j)$$

Question 7.2 (10 points)

In the cell below, please enter the code to print the NDCG@k where k={5, 10} for the input ranking file. As mentioned above, please note that the final value is the average of metric values across all queries.

Use log base 2 for the calculations.


In [ ]:
import numpy as np
'''
In this function, calculate NDCG@k, given the input ranking information (rank_in)
and the query passage relevance information (qrels).
Return Value:
ndcg - NDCG@k
'''
def calcNDCG(k, qrels, rank_in):
  # data structure init
  dcg_array = [0] * len(rank_in)
  idcg_array = [0] * len(rank_in)
  ndcg_array = [0] * len(rank_in)

  for idx, query in enumerate(rank_in):
    passages = rank_in[query]
    passages_rank = qrels[query]

    # Calculate DCG:
    dcg = 0
    for idxPassage, passage in enumerate(passages[:k]):
      if passage in passages_rank:
        dcg += (passages_rank[passage] - 1) / (np.log2(idxPassage + 2))

    dcg_array[idx] = dcg

    # calculate IDCG:
    idcg = 0
    # get passages from qrels[query]
    # we sort by value (rank)
    sorted_rank_passages = sorted(passages_rank.items(), key=lambda item: -item[1])

    for idxPassage, passage in enumerate(sorted_rank_passages[:k]):
      idcg += (passage[1] - 1) / (np.log2(idxPassage + 2))

    idcg_array[idx] = idcg

    if idcg == 0:
      ndcg_array[idx] = 0
    else:
      ndcg_array[idx] = dcg / idcg

  ndcg = sum(ndcg_array) / len(ndcg_array)

  return ndcg

print ('NDCG at top 5 : {0}'.format(calcNDCG(5, qrels, rank_in)))
print ('NDCG at top 10 : {0}'.format(calcNDCG(10, qrels, rank_in)))

NDCG at top 5 : 0.19759308493592626
NDCG at top 10 : 0.1900647292491866
